# Step 9: Make the Assam images and the U-Net models viewable

**Run in Google Colab** (*Runtime > Run all*). No GPU needed. About 10 minutes.

**Before running, in your Google Drive (My Drive):**
- `cutoff_assam`: the three `.tif` files from Step 5 (already there)
- `cutoff_models`: make this folder and upload the five files `unet_seed0.pth` to `unet_seed4.pth`

**What you get** in `My Drive/cutoff_viewable` (and as one zip to download):

| File | Open with | What it shows |
|---|---|---|
| `*.png` | Any image viewer | The image with a title and a one-line key |
| `*.kmz` | Google Earth (desktop or earth.google.com > Projects > Import) | The same image laid on the real map |
| `unet_seedN_summary.txt` | Any text editor | The network's parts and every weight's shape |
| `unet_seedN.onnx` | Drag onto **netron.app** in a browser | The whole U-Net drawn as a diagram |

The pictures are shrunk to about 4,000 pixels across (about 35 m per pixel) so they open quickly. The original `.tif` files stay the master copy.

In [ ]:
!pip install -q segmentation-models-pytorch onnx rasterio > /dev/null

In [ ]:
import os, sys, subprocess, glob, shutil
REPO = "/content/cutoff-flood"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/Matrixco123/cutoff-flood.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
sys.path.insert(0, REPO)
from src import viewable as V
import numpy as np

from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive"
IN_TIF, IN_MOD, OUT = f"{D}/cutoff_assam", f"{D}/cutoff_models", "/content/cutoff_viewable"
os.makedirs(OUT, exist_ok=True)
print("tifs:", sorted(os.listdir(IN_TIF)) if os.path.exists(IN_TIF) else "MISSING")
print("models:", sorted(glob.glob(f"{IN_MOD}/*.pth")) if os.path.exists(IN_MOD) else "MISSING")

## 1. Radar images (17 and 29 July)

In [ ]:
NOD = -32768
for date, f in [("17 July (before)", "s1_pre_20260717"), ("29 July (during)", "s1_post_20260729")]:
    path = f"{IN_TIF}/{f}.tif"
    V.convert(path, OUT, f"{f}_VV", f"Radar {date}, VV", "Black = calm water, grey = land and fields, white = buildings. Scale -25 to 0 dB.",
              band=1, scale=0.01, nodata=NOD, vmin=-25, vmax=0, cmap="gray")
    V.convert(path, OUT, f"{f}_VH", f"Radar {date}, VH", "Black = calm water, grey = land, white = buildings and trees. Scale -30 to -5 dB.",
              band=2, scale=0.01, nodata=NOD, vmin=-30, vmax=-5, cmap="gray")
    # false colour: red = VV, green = VH, blue = VV minus VH. Water = black, fields = green-blue, towns = bright
    vv, tr, crs = V.read_band(path, 1, 0.01, NOD); vh, _, _ = V.read_band(path, 2, 0.01, NOD)
    def st(x, lo, hi): return np.clip((x - lo) / (hi - lo), 0, 1)
    rgb = np.dstack([st(vv, -25, 0), st(vh, -30, -5), st(vv - vh, 2, 15)])
    rgba = (np.dstack([rgb, ~np.isnan(vv)]) * 255).astype(np.uint8)
    rgba[np.isnan(vv)] = 0
    V.save_png_with_key(rgba, f"{OUT}/{f}_falsecolour.png", f"Radar {date}, false colour",
                        "Red = VV, green = VH, blue = VV minus VH. Black = water, bright = towns.")
    V.save_kmz(rgba, tr, crs, f"{OUT}/{f}_falsecolour.kmz", f"Radar {date}, false colour")
    print("done", f)

## 2. Permanent water and terrain height

In [ ]:
aux = f"{IN_TIF}/aux_jrc_hand.tif"
V.convert(aux, OUT, "jrc_permanent_water", "How often each place was water, 1984 to 2021 (JRC)",
          "Light blue = sometimes water, dark blue = almost always. Transparent = never water.",
          band=1, vmin=1, vmax=100, cmap="Blues", zero_transparent=True)
V.convert(aux, OUT, "hand_height_above_stream", "Height above the nearest stream (MERIT HAND)",
          "Dark = low, close to streams (can flood). Bright = more than 50 m above (cannot flood). We exclude above 15 m.",
          band=2, scale=0.1, nodata=NOD, vmin=0, vmax=50, cmap="viridis")
print("done")

## 3. The five U-Net models

In [ ]:
for p in sorted(glob.glob(f"{IN_MOD}/*.pth")):
    print(V.summarise_unet(p, OUT))
print(open(f"{OUT}/unet_seed0_summary.txt").read()[:900] if os.path.exists(f"{OUT}/unet_seed0_summary.txt") else "no models found")

## 4. Save to Drive and download

In [ ]:
shutil.copytree(OUT, f"{D}/cutoff_viewable", dirs_exist_ok=True)
small = [f for f in os.listdir(OUT) if not f.endswith(".onnx")]
shutil.make_archive("/content/cutoff_viewable_pictures", "zip", OUT, ".")
for f in sorted(os.listdir(OUT)):
    print(f"{f:45s} {os.path.getsize(f'{OUT}/{f}') / 1e6:7.1f} MB")
print("\nCopied to My Drive/cutoff_viewable")
from google.colab import files
files.download("/content/cutoff_viewable_pictures.zip")